In [1]:
import pandas as pd
import numpy as np

DATA_FILE      = "DISSERTATION_FULL_DATA_FINAL.xlsx"
BAA_FILE       = "BAA10Y.xlsx"
FILL_THRESHOLD = 5
SAMPLE_START   = "2010-01-01"
SAMPLE_END     = "2024-12-31"

SECTOR_CONFIG = {
    "XLF 5Y FINANCIALS": {"code": "XLF", "tickers": {
        "JPMORGAN": "JPM", "JP MORGAN": "JPM", "BERKSHIRE": "BRK",
        "BANK OF AMERICA": "BAC", "BOFA": "BAC", "GOLDMAN": "GS",
        "MORGAN STANLEY": "MS", "WELLS FARGO": "WFC", "CITIGROUP": "C",
        "CITI": "C", "AMERICAN EXPRESS": "AXP", "CAPITAL ONE": "COF",
        "CHUBB": "CB"}},
    "XLI 5Y INDUSTRIALS": {"code": "XLI", "tickers": {
        "RAYTHEON": "RTX", "EATON": "ETN", "DEERE": "DE", "LOCKHEED": "LMT",
        "GENERAL DYNAMICS": "GD", "3M": "MMM", "HONEYWELL": "HON",
        "BOEING": "BA", "CATERPILLAR": "CAT", "UNION PACIFIC": "UNP"}},
    "XLV 5Y HEALTHCARE": {"code": "XLV", "tickers": {
        "ELI LILLY": "LLY", "LILLY": "LLY", "JOHNSON": "JNJ", "J&J": "JNJ",
        "UNITEDHEALTH": "UNH", "UNITED HEALTH": "UNH", "MERCK": "MRK",
        "CVS": "CVS", "MEDTRONIC": "MDT", "HUMANA": "HUM", "AMGEN": "AMGN",
        "PFIZER": "PFE", "BRISTOL": "BMY"}},
    "XLY 5Y CONSUMER DISCRETIONARY": {"code": "XLY", "tickers": {
        "TJX": "TJX", "MARRIOTT": "MAR", "NIKE": "NKE", "EXPEDIA": "EXPE",
        "ROYAL CARIBBEAN": "RCL", "ROYAL": "RCL", "MGM": "MGM",
        "CARNIVAL": "CCL", "HOME DEPOT": "HD", "MCDONALD": "MCD",
        "LOWE'S": "LOW", "LOWES": "LOW"}},
    "XLK 5Y TECHNOLOGY": {"code": "XLK", "tickers": {
        "MICROSOFT": "MSFT", "INTEL": "INTC", "TEXAS INSTRUMENTS": "TXN",
        "ORACLE": "ORCL", "SEAGATE": "STX", "CORNING": "GLW", "NXP": "NXPI",
        "MOTOROLA": "MSI", "IBM": "IBM", "APPLE": "AAPL"}},
    "XLE 5Y ENERGY": {"code": "XLE", "tickers": {
        "EXXON": "XOM", "CHEVRON": "CVX", "WILLIAMS": "WMB", "EOG": "EOG",
        "ONEOK": "OKE", "OCCIDENTAL": "OXY"}}, }



In [2]:
def find_data_start(df):
    for i in range(8):
        try:
            if pd.notna(pd.to_datetime(df.iloc[i, 0])):
                return i
        except (ValueError, TypeError):
            pass
    raise ValueError("Could not locate data start.")

def map_columns(df, ticker_map):
    header = [str(df.iloc[0, c]).strip().upper() for c in range(df.shape[1])]
    keys_by_length = sorted(ticker_map, key=len, reverse=True)
    col_to_ticker, used = {}, set()
    for col, name in enumerate(header):
        if name in ("", "NAN", "DATE", "NONE"):
            continue
        for key in keys_by_length:
            ticker = ticker_map[key]
            if ticker not in used and key.upper() in name:
                col_to_ticker[col] = ticker
                used.add(ticker)
                break
    return col_to_ticker
# Gap audit to check which will be filled and which will be excluded
def count_gaps(series):
    mask = series.isna()
    if not mask.any():
        return {"Forward_Fill_Block": 0, "Exclude_Block": 0, "Total_NaN_Days": 0}
    block_sizes = mask.ne(mask.shift()).cumsum()[mask].value_counts()
    return {
        "Forward_Fill_Block":   int((block_sizes <= FILL_THRESHOLD).sum()),
        "Exclude_Block":   int((block_sizes >  FILL_THRESHOLD).sum()),
        "Total_NaN_Days": int(mask.sum()),}

In [3]:
all_sheets = pd.read_excel(DATA_FILE, sheet_name=None, header=None)

In [4]:
firm_cds = []
for sheet, cfg in SECTOR_CONFIG.items():
    if sheet not in all_sheets:
        continue
    sector = cfg["code"]
    raw = all_sheets[sheet].dropna(axis=1, how="all")
    start = find_data_start(raw)
    col_map = map_columns(raw, cfg["tickers"])
    rows = raw.iloc[start:]
    dates = pd.to_datetime(rows.iloc[:, 0], errors="coerce")
    rows, dates = rows[dates.notna()], dates[dates.notna()]
    for col, ticker in col_map.items():
        s = pd.to_numeric(rows.iloc[:, col], errors="coerce")
        s.index = dates
        s = s.sort_index().loc[SAMPLE_START:SAMPLE_END]
        # Gaps: forward-fill gaps up to one week, keep the rest NaN
        s = s.ffill(limit=FILL_THRESHOLD)
        firm_cds.append(pd.DataFrame({
            "Date":          s.index,
            "Firm":          ticker,
            "Sector":        sector,
            "CDS_level":     s.values,
            "CDS_change_5d": s.diff(5).values,
        }))

cds = pd.concat(firm_cds, ignore_index=True).dropna(subset=["CDS_level"])

In [5]:
raw = all_sheets["MACRO CONTROLS"].dropna(axis=1, how="all")
start = find_data_start(raw)
rows = raw.iloc[start:]
dates = pd.to_datetime(rows.iloc[:, 0], errors="coerce")
rows, dates = rows[dates.notna()], dates[dates.notna()]
macro = pd.DataFrame({
    "VIX":   pd.to_numeric(rows.iloc[:, 1], errors="coerce").values,
    "US2Y":  pd.to_numeric(rows.iloc[:, 2], errors="coerce").values,
    "US10Y": pd.to_numeric(rows.iloc[:, 3], errors="coerce").values,
}, index=pd.to_datetime(dates.values))
macro = macro.sort_index().loc[SAMPLE_START:SAMPLE_END]
macro["Yield_slope"] = macro["US10Y"] - macro["US2Y"]

baa = pd.read_excel(BAA_FILE, sheet_name="Daily", header=0)
baa.columns = ["Date", "BAA10Y"]
baa["Date"] = pd.to_datetime(baa["Date"])
baa["BAA10Y"] = pd.to_numeric(baa["BAA10Y"], errors="coerce")
baa = baa.set_index("Date").sort_index().loc[SAMPLE_START:SAMPLE_END]

macro = macro.join(baa, how="left")
for col in ["VIX", "Yield_slope", "BAA10Y"]:
    macro[col] = macro[col].ffill(limit=FILL_THRESHOLD)
macro.index.name = "Date"
macro = macro.reset_index()
macro = macro[["Date", "VIX", "Yield_slope", "BAA10Y"]]

In [6]:
cds.to_parquet("Firm_CDS.parquet", index=False)
macro.to_parquet("Macro_Controls.parquet", index=False)